In [24]:
%pip install docent-python datasets pandas python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
from datasets import load_dataset

# Load the Verified dataset
# Note: Verified usually uses the 'test' split for evaluation
dataset = load_dataset("SWE-bench/SWE-bench_Verified", split="test")

# Convert to a dictionary for easy lookup by instance_id
ground_truth = {ins["instance_id"]: ins["patch"] for ins in dataset}


In [26]:
HARD_PROBLEMS = [
    # Pattern 1: Infinite loop / churn spiral (7) — hit $3 cost ceiling, never converged
    "django__django-10554",
    "django__django-11265",
    "django__django-11734",
    "django__django-13344",
    "django__django-15554",
    "django__django-16263",
    "sphinx-doc__sphinx-9229",

    # Pattern 2: Pylint blind spot (8) — 80% fail rate, lacks AST-visitor architecture insight
    "pylint-dev__pylint-4551",
    "pylint-dev__pylint-4604",
    "pylint-dev__pylint-4661",
    "pylint-dev__pylint-6386",
    "pylint-dev__pylint-6528",
    "pylint-dev__pylint-7080",
    "pylint-dev__pylint-7277",
    "pylint-dev__pylint-8898",

    # Pattern 3: Sphinx build maze (11) — lost in deeply-coupled extension system, avg cost $1.80
    "sphinx-doc__sphinx-10435",
    "sphinx-doc__sphinx-10614",
    "sphinx-doc__sphinx-11510",
    "sphinx-doc__sphinx-7462",
    "sphinx-doc__sphinx-7590",
    "sphinx-doc__sphinx-7748",
    "sphinx-doc__sphinx-7985",
    "sphinx-doc__sphinx-8548",
    "sphinx-doc__sphinx-9461",
    "sphinx-doc__sphinx-9602",

    # Pattern 4: Overconfident quick fail (5 unique) — <20 API calls, skipped verification
    "django__django-10999",
    "django__django-14140",
    "django__django-14315",
    # (pylint-4661 and pylint-7277 also here but listed in Pattern 2)

    # Pattern 5: Sympy math reasoning gap (21) — lacks domain-specific symbolic math insight
    "sympy__sympy-13031",
    "sympy__sympy-13798",
    "sympy__sympy-13852",
    "sympy__sympy-13877",
    "sympy__sympy-13974",
    "sympy__sympy-14248",
    "sympy__sympy-14976",
    "sympy__sympy-15599",
    "sympy__sympy-15875",
    "sympy__sympy-16597",
    "sympy__sympy-17318",
    "sympy__sympy-17630",
    "sympy__sympy-18199",
    "sympy__sympy-18763",
    "sympy__sympy-19495",
    "sympy__sympy-20428",
    "sympy__sympy-20438",
    "sympy__sympy-21596",
    "sympy__sympy-21930",
    "sympy__sympy-22080",
    "sympy__sympy-23950",

    # Pattern 6: High-churn wrong answer (10 unique) — >$1.50 cost, submitted wrong, tunnel-visioned
    "astropy__astropy-13398",
    "astropy__astropy-14598",
    "django__django-12663",
    "django__django-14034",
    "django__django-15957",
    "django__django-15973",
    "matplotlib__matplotlib-25311",
    "matplotlib__matplotlib-26208",
    "pydata__xarray-6599",
    "pydata__xarray-7229",
]

In [ ]:
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Point this at your agent-runs export (string or Path; relative paths use the notebook/kernel cwd).
CSV_FILEPATH = Path("./agent-runs/csvs/agent-runs-b038912e-0133-4594-b093-92806f8ffb17 (1).csv")

load_dotenv()

csv_path = Path(CSV_FILEPATH).expanduser().resolve()
if not csv_path.is_file():
    raise FileNotFoundError(f"CSV not found: {csv_path}")

REPO_ROOT = csv_path.parent
print(f"Using CSV: {csv_path}")

df = pd.read_csv(csv_path)
run_id_map = df.set_index("metadata.instance_id")["agent_run_id"].to_dict()


Using CSV: C:\Users\coler\OneDrive\Documents\swe-bench-research\agent-runs-csvs\agent-runs-b038912e-0133-4594-b093-92806f8ffb17 (1).csv


In [28]:
import os

from docent import Docent

DOCENT_API_KEY = os.getenv("DOCENT_API_KEY")
if not DOCENT_API_KEY:
    raise ValueError(
        "DOCENT_API_KEY is missing. Set it in a .env file in the project root "
        "(see .env.example) or export it in your environment."
    )

client = Docent(api_key=DOCENT_API_KEY)
COLLECTION_ID = "b038912e-0133-4594-b093-92806f8ffb17"


def _message_content_to_str(content) -> str:
    """Normalize Docent ChatMessage content (str or list of Content blocks) to a string."""
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    parts = []
    for block in content:
        if isinstance(block, str):
            parts.append(block)
        elif hasattr(block, "text") and block.text is not None:
            parts.append(str(block.text))
        elif hasattr(block, "reasoning") and block.reasoning is not None:
            parts.append(str(block.reasoning))
        elif isinstance(block, dict):
            parts.append(str(block.get("text") or block.get("reasoning") or block))
        else:
            parts.append(str(block))
    return "\n".join(parts)


def get_model_attempt(instance_id):
    """Returns the model's full attempt transcript as a string for a given instance_id.

    Uses client.get_agent_run(), which returns full Transcript objects with `messages`
    (chat turns). The Docent SDK does not provide get_transcript(); see:
    https://docs.transluce.org/sdk/agent-runs/query
    """
    agent_run_id = run_id_map.get(instance_id)
    if not agent_run_id:
        return None

    try:
        run = client.get_agent_run(COLLECTION_ID, agent_run_id)
        if run is None:
            return f"Error: no agent run found for {instance_id} (id={agent_run_id})"
        if not run.transcripts:
            return f"Error: agent run {agent_run_id} has no transcripts"

        output = [f"## Instance: {instance_id} (Run: {agent_run_id})\n"]

        for ti, transcript in enumerate(run.transcripts):
            if len(run.transcripts) > 1:
                tid = getattr(transcript, "id", None)
                output.append(f"### Transcript {ti + 1} (id={tid})\n")

            messages = getattr(transcript, "messages", None) or []
            for i, msg in enumerate(messages):
                role = getattr(msg, "role", "unknown")
                output.append(f"### Message {i + 1} ({role})\n")

                body = _message_content_to_str(getattr(msg, "content", None))
                if body:
                    shown = body if len(body) <= 12000 else body[:12000] + "\n\n… [truncated]"
                    output.append(shown)

                tool_calls = getattr(msg, "tool_calls", None)
                if tool_calls:
                    for tc in tool_calls:
                        fn = getattr(tc, "function", None)
                        args = getattr(tc, "arguments", None)
                        tid = getattr(tc, "id", None)
                        args_s = args if isinstance(args, str) else repr(args)
                        if len(args_s) > 4000:
                            args_s = args_s[:4000] + "…"
                        output.append(
                            f"\n**Tool call** `{fn}` (id={tid}):\n```\n{args_s}\n```"
                        )

                if role == "tool":
                    tname = getattr(msg, "name", None)
                    tcid = getattr(msg, "tool_call_id", None)
                    if tname or tcid:
                        output.append(f"\n*(tool name={tname}, tool_call_id={tcid})*")

                output.append("\n---\n")

        return "\n".join(output)
    except Exception as e:
        return f"Error retrieving transcript for {instance_id}: {str(e)}"

13:53:39 [INFO] docent.sdk._base: Authenticating Docent client with frontend_url='https://docent.transluce.org' and api_url='https://api.docent.transluce.org/rest'
13:53:41 [INFO] docent.sdk._base: Logged in with API key


In [ ]:
missing_instance_ids = []

def write_comparison_file():
    out_dir = REPO_ROOT / "agent-runs" / "outputs"
    out_dir.mkdir(parents=True, exist_ok=True)
    filename = out_dir / f"{COLLECTION_ID}_comparison.md"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"# Comparison Report for Collection {COLLECTION_ID}\n\n")

        for instance_id in HARD_PROBLEMS:
            # Check if we have both ground truth and a run_id
            has_gt = instance_id in ground_truth
            has_run = instance_id in run_id_map

            if not has_gt or not has_run:
                missing_instance_ids.append(instance_id)
                continue

            f.write(f"# INSTANCE ID: {instance_id}\n\n")

            # Write Model Attempt
            f.write("## MODEL ATTEMPT\n")
            attempt_text = get_model_attempt(instance_id)
            if attempt_text:
                f.write(attempt_text + "\n")
            else:
                f.write("Failed to retrieve attempt.\n")

            # Write Ground Truth
            f.write("\n## GROUND TRUTH PATCH\n")
            f.write(f"```diff\n{ground_truth[instance_id]}\n```\n")
            f.write("\n" + "="*80 + "\n\n")

    print(f"Comparison file written to: {filename}")
    if missing_instance_ids:
        print(f"Missing data for: {missing_instance_ids}")

write_comparison_file()

Comparison file written to: C:\Users\coler\OneDrive\Documents\swe-bench-research\agent-runs-csvs\b038912e-0133-4594-b093-92806f8ffb17_comparison.md
